# Calculate the relative binding free energy of peptides using the OpenFF Rosemary force field

This tutorial demonstrates how we can use the OpenFF Rosemary force field with AshGC partial charges to more accurately model the peptides (keeping AMBER FF14SB for the protein receptor).

As an example, we calculate the relative binding free energy of two peptides (``1AOF`` and ``1AOM``) bound to ``TIAM1``. These systems are taken from 
["Relative Binding Free Energy Estimation of Congeneric Ligands and Macromolecular Mutants with the Alchemical Transfer Method with Coordinate Swapping"](https://doi.org/10.1021/acs.jcim.5c00207) by Emilio Gallichio.

## Loading the ligands

Whilst OpenFE can read peptides when parsed through a `ProteinComponent`, this route provides inssuficient information (e.g. bond orders) to be able to assign SMIRNOFF parameters. Instead, we must load the peptides into `SmallMoleculeComponent`s. Here we have peptides in `SDF` files, and we load them directly using RDKit.

In [3]:
from rdkit import Chem
import openfe

In [4]:
# Load 1AOF
rdmol = [m for m in Chem.SDMolSupplier('assets/peptides/1AOF.sdf', removeHs=False)][0]
smc_1aof = openfe.SmallMoleculeComponent(rdmol)

In [5]:
# Load 1AOM
rdmol = [m for m in Chem.SDMolSupplier('assets/peptides/1AOM.sdf', removeHs=False)][0]
smc_1aom = openfe.SmallMoleculeComponent(rdmol)

## Assigning partial charges with AshGC

Traditional partial charge methods for small molecules (i.e. AM1BCC) are unreasonable for use with larger systems such as peptides. To this end, the Rosemary force field was instead fitted to use AshGC charges, a graph neural network trained to produce conformer-independent charges of semi-empirical quality at linear cost for molecules of all sizes.

Here we will apply AshGC partial charges to both small molecules.

In [6]:
from openfe.protocols.openmm_utils.charge_generation import assign_offmol_partial_charges

In [7]:
# Charge 1AOF
# first convert to an openff molecule, then assign partial charges
# then we will recreate the SmallMoleculeComponent
offmol = smc_1aof.to_openff()
offmol = assign_offmol_partial_charges(
    offmol,
    overwrite=True,
    method="nagl",
    toolkit_backend="rdkit",
    generate_n_conformers=None,
    nagl_model="openff-gnn-am1bcc-1.0.0.pt"
)
smc_1aof = openfe.SmallMoleculeComponent.from_openff(offmol)

In [8]:
# Now let's do the same of 1AOM
offmol = smc_1aom.to_openff()
offmol = assign_offmol_partial_charges(
    offmol,
    overwrite=True,
    method="nagl",
    toolkit_backend="rdkit",
    generate_n_conformers=None,
    nagl_model="openff-gnn-am1bcc-1.0.0.pt"
)
smc_1aom = openfe.SmallMoleculeComponent.from_openff(offmol)

## Creating a ligand atom mapping for the peptides

In this demonstration, we will be using OpenFE's hybrid topology method to calculate the relative binding free energy between the two peptides. To create the hybrid system, we need to create an atom mapping which defines which atoms are common and unique between the two end state molecules.

To achieve this, we will be using the Kartograf atom mapper with its default settings.

In [9]:
mapper = openfe.KartografAtomMapper()
mapping = next(mapper.suggest_mappings(smc_1aom, smc_1aof))

In [10]:
mapping.view_3d()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

From this mapping, we can see that the PHE and MET residues are unique to each end state whilst the remaining atoms are shared.

## Creating the Hybrid Topology ``Transformation``s

Next we follow a similar approach to the [RBFE Python Tutorial](https://docs.openfree.energy/en/stable/tutorials/rbfe_python_tutorial.html) to create a complex and solvent ``Transformation`` between these two molecules.

### Creating the ``ChemicalSystems``

First we start by creating the ``ChemicalSystems`` that define the endstates of the ``Transformation``, both with and without the present of the ``TIAM1`` protein.

In [11]:
# Load in the TIAM1 protein
protein = openfe.ProteinComponent.from_pdb_file("assets/peptides/protein.pdb")

In [12]:
# Create the ChemicalSystems for 1AOF

system_1aof_complex = openfe.ChemicalSystem({"protein": protein, "ligand": smc_1aom, "solvent": openfe.SolventComponent()})
system_1aof_solvent = openfe.ChemicalSystem({"ligand": smc_1aom, "solvent": openfe.SolventComponent()})

In [13]:
# Create the CHemicalSystems for 1AOM

system_1aom_complex = openfe.ChemicalSystem({"protein": protein, "ligand": smc_1aof, "solvent": openfe.SolventComponent()})
system_1aom_solvent = openfe.ChemicalSystem({"ligand": smc_1aof, "solvent": openfe.SolventComponent()})

## Creating the Protocol settings

Now let's create settings for the `RelativeHybridTopologyProtocol`s for both the solvent and complex `Transformation`s.

In [14]:
from openfe.protocols.openmm_rfe import RelativeHybridTopologyProtocol

settings = RelativeHybridTopologyProtocol.default_settings()
# Set the small molecule force field (e.g. the one applied to the peptides) to be the Alpha release of the Rosemary force field
settings.forcefield_settings.small_molecule_forcefield = "openff_no_water-3.0.0-alpha0.offxml"
# Set the protein force field to be ff14SB
settings.forcefield_settings.forcefields = [
    "amber/ff14SB.xml",  # ff14SB protein force field
    "amber/tip3p_standard.xml",  # TIP3P and recommended monovalent ion parameters
]
# We set the number of repeats to 1 so that we can execute repeats separately
settings.protocol_repeats = 1

solvent_settings = RelativeHybridTopologyProtocol._adaptive_settings(stateA=system_1aof_solvent, stateB=system_1aom_solvent, mapping=mapping)
complex_settings = RelativeHybridTopologyProtocol._adaptive_settings(stateA=system_1aof_complex, stateB=system_1aom_complex, mapping=mapping)

## Creating the `Protocol`s

We next create Protocols for both the solvent and complex legs.

In [15]:
solvent_protocol = RelativeHybridTopologyProtocol(settings=solvent_settings)
complex_protocol = RelativeHybridTopologyProtocol(settings=complex_settings)

## Creating the ``Transformation``s

Finally we create the ``Transformation``s and write them to disk for later execution.

In [16]:
# Create the Transformations
solvent_transform = openfe.Transformation(
    stateA=system_1aof_solvent,
    stateB=system_1aom_solvent,
    mapping=mapping,
    protocol=solvent_protocol,
    name="solvent 1AOF to 1AOM transformation",
)
complex_transform = openfe.Transformation(
    stateA=system_1aof_complex,
    stateB=system_1aom_complex,
    mapping=mapping,
    protocol=complex_protocol,
    name="complex 1AOF to 1AOM transformation",
)

In [17]:
# Write the Transformations to disk
solvent_transform.to_json("solvent_transform.json")
complex_transform.to_json("complex_transform.json")

## Running the transformations and next steps

We can run these transformations using `quickrun`, please see our [documentation on how to do this](https://docs.openfree.energy/en/stable/guide/execution/quickrun_execution.html).

If you want to scale this tutorial to a whole network of peptide transformations, you can apply the settings / charge methods demonstrated here to the [tutorial on setting up a relative binding free energy network](https://docs.openfree.energy/en/stable/tutorials/rbfe_python_tutorial.html). 